# Import

In [1]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from pybounds import Simulator, SlidingEmpiricalObservabilityMatrix, FisherObservability, SlidingFisherObservability, ObservabilityMatrixImage, colorline


C:\Users\mayc06\AppData\Local\anaconda3\envs\CelliniWindSim\Lib\site-packages\do_mpc\sysid\__init__.py:15: UserWarning: The ONNX feature is not available. Please install the full version of do-mpc to access this feature.
  warnings.warn('The ONNX feature is not available. Please install the full version of do-mpc to access this feature.')
C:\Users\mayc06\AppData\Local\anaconda3\envs\CelliniWindSim\Lib\site-packages\do_mpc\opcua\__init__.py:14: UserWarning: The opcua feature is not available. Please install the full version of do-mpc to access this feature.
  warnings.warn('The opcua feature is not available. Please install the full version of do-mpc to access this feature.')


# Measure observability of each trajectory

The simulator object defines the dynamics that govern the activity in the PFNs, allowing the pybounds package to use empirical methods to measure observability. So, you can't just feed this the PFN activity you already computed. You have to build everything anew. (I think.)

### First define the measurement and dynamics functions

In [2]:
state_names = ['x',  # x position [m]
               'y',  # y position [m]
               'z',  # z position (altitude) [m]
               'v_para',  # parallel velocity [m/s]
               'v_perp',  # perpendicular velocity [m/s]
               'phi', # heading [rad]
               'phi_dot',  # angular velocity [rad/s]
               'w',  # ambient wind speed [m/s]
               'zeta',  # ambient wind angle [rad]
               'm', # mass [kg]
               'I',  # inertia [kg*m^2]
               'C_para',  # parallel damping [N*s/m]
               'C_perp',  # perpendicular damping [N*s/m]
               'C_phi',  # rotational damping [N·m/rad/s]
               'km1',  # parallel motor calibration coefficient
               'km2',  # offset motor calibration coefficient
               'km3',  # perpendicular motor calibration coefficient
               'km4',  # rotational motor calibration coefficient
               ]

input_names = ['u_para',  # parallel thrust force [N]
               'u_perp',  # perpendicular thrust force [N]
               'u_phi',  # torque [N*m]
               'u_w',                                                            ####
               'u_zeta'                                                          ####
               ]
def f(X, U):
    # States
    x, y, z, v_para, v_perp, phi, phi_dot, w, zeta, m, I, C_para, C_perp, C_phi, km1, km2, km3, km4 = X

    # Inputs
    #u_para, u_perp, u_phi = U
    u_para, u_perp, u_phi, u_w, u_zeta = U                                       ####

    # Air velocity
    a_para = v_para - w * np.cos(phi - zeta)
    a_perp = v_perp + w * np.sin(phi - zeta)

    # Acceleration
    v_para_dot = ((km1 * u_para - C_para * a_para) / m) + (v_perp * phi_dot)
    v_perp_dot = ((km3 * u_perp - C_perp * a_perp) / m) - (v_para * phi_dot)

    # Angular acceleration
    phi_ddot = (km4 * u_phi / I) - (C_phi * phi_dot / I) + (km2 * u_para / I)

    # Other dynamics
    x_dot = v_para * np.cos(phi) - v_perp * np.sin(phi)
    y_dot = v_para * np.sin(phi) + v_perp * np.cos(phi)
    z_dot = 0*x
    w_dot = 0*x
    #w_dot = u_w                                                                   ####
    #zeta_dot = 0*x
    zeta_dot = u_zeta                                                             ####
    m_dot = 0*x
    I_dot = 0*x
    C_para_dot = 0*x
    C_perp_dot = 0*x
    C_phi_dot = 0*x
    km1 = 0*x
    km2 = 0*x
    km3 = 0*x
    km4 = 0*x

    # Package and return xdot
    x_dot = [x_dot, y_dot, z_dot, v_para_dot, v_perp_dot, phi_dot, phi_ddot, w_dot, zeta_dot, m_dot, I_dot, C_para_dot, C_perp_dot, C_phi_dot, km1, km2, km3, km4]

    return x_dot

In [3]:
measurement_names = ['phi', 'psi', 'gamma', 'a', 'g', 'r']
def h(X, U):
    # States
    x, y, z, v_para, v_perp, phi, phi_dot, w, zeta, m, I, C_para, C_perp, C_phi, km1, km2, km3, km4 = X

    # Inputs
    #u_para, u_perp, u_phi = U
    u_para, u_perp, u_phi, u_w, u_zeta = U                                     ####

    # Air velocity ### Is this in world coordinates?
    a_para = v_para - w * np.cos(phi - zeta)  # in fly ref frame
    a_perp = v_perp + w * np.sin(phi - zeta)  # in fly ref frame
    a = np.sqrt(a_para ** 2 + a_perp ** 2)
    gamma = np.arctan2(a_perp, a_para)  # air velocity angle   ### in fly ref frame?

    # Course direction in fly reference frame
    g = np.sqrt(v_para ** 2 + v_perp ** 2)
    psi = np.arctan2(v_perp, v_para)

    # Optic flow
    r = g / z

    # Unwrap angles
    if np.array(phi).ndim > 0:
        if np.array(phi).shape[0] > 1:
            phi = np.unwrap(phi)
            psi = np.unwrap(psi)
            gamma = np.unwrap(gamma)

    # Measurements
    Y  = [phi, psi, gamma, a, g, r]

    # Return measurement
    return Y


### Initialize trajectory parameters

These are for non-saccade turn trajectories (i.e., bio validation stimuli with varying control).

- 1 deceleration trajectory that emulates dynamics of real deceleration
- 1 decel + non-saccade turn to the right
- 1 decel + non-saccade turn to the left

In [4]:
fs = 50   # 500 for Mayetal2025
dt = 1 / fs
T = 12.0  #1.0 for complex-traj 01:08
tsim = np.arange(dt, T + 0.9*dt, dt)

v_para_0 = [0.2]
v_para_dot_0 = [2,1]
v_perp_0 = 0.001
v_perp_dot_0 = 0.00001

initphi = [0]
phi_0 = [0.02]

#deltaPhi = [0.0]  # decel-only
deltaPhi = [-np.pi/2]  # turnleft (+) or turnright (-)

zeta_0 = list(np.arange(np.pi/4,2*np.pi,np.pi/2) - np.pi/5)

w_0 = [0.45]
control_w = 0.3

elev = [0.3]

### Calculate trajectory dynamics using MPC, create simulator object, and measure observability for each trajectory.

In [ ]:
## Simple, non-saccade trajectories

obs_stimulus = pd.DataFrame()

obs_stimfilename = 'decel-turnright_multiZeta_noHeadingPredictor_dt0-02_30cmscontrol_observability'
obs_stimfilename = obs_stimfilename+'.csv'
obs_stimfile = open(obs_stimfilename, 'a')


count = 0
n_traj = len(zeta_0)
for zed in zeta_0:
    
    simulator = Simulator(f, h, dt=dt, 
                          state_names=state_names, 
                          input_names=input_names, 
                          measurement_names=measurement_names,
                          mpc_horizon=10
                         )
    
    # include straight-trajectory baseline for first 2 s to handle MPC initial dynamics
    zeta = zed*np.ones((int(fs*12),))
    w = w_0[0]*np.ones((int(fs*12),))
    v_perp = v_perp_0*np.ones((int(fs*12),)) + v_perp_dot_0*tsim
    v_para = v_para_0*np.ones((int(fs*5),))
    v_para = np.append(v_para,v_para[-1]-(0.1*(1/(1+np.exp(6*(-v_para_dot_0[0]*tsim[:fs*2]+1))))))
    v_para = np.append(v_para,v_para[-1]+(0.1*(1/(2+np.exp(6*(-v_para_dot_0[1]*tsim[:fs*2]+1))))))
    v_para = np.append(v_para,v_para[-1]*np.ones((int(fs*3),)))
    
    #phi = phi_0*np.ones((int(fs*12),))   # decel-only
    
    phi = phi_0*np.ones((int(fs*4.5),))    # fs*4.5 normal, fs*3.5 slow
    angaccel = (deltaPhi)/(1+np.exp(-3*(1*np.arange(0,4.5,dt)-2)))  # normal
    ##angaccel = (deltaPhi)/(1+np.exp(-2*(1*np.arange(0,6.5,dt)-3)))  # slow
    phi = np.append(phi,phi[-1]+angaccel)
    phi = np.append(phi,phi[-1]*np.ones((int(fs*3),)))   # fs*3 normal, fs*2 slow
       
    alt = elev*np.ones((int(fs*12),))
    
    print(len(alt))
    print(T)
    setpoint_CEM = {'x': 0.0 * np.ones_like(alt),
                    'y': 0.0 * np.ones_like(alt),
                    'z': alt,
                    'v_para': v_para,
                    'v_perp': v_perp,
                    'phi': phi,
                    'phi_dot': 0.0*np.ones_like(alt),
                    'w': 0.0*np.ones_like(alt),         ###
                    #'w': w_0[0]*np.ones_like(alt),
                    #'zeta': 0.0*np.ones_like(alt),
                    'zeta': zeta,                       ###
                    'm': m * np.ones_like(alt),
                    'I': I * np.ones_like(alt),
                    'C_para': C_para * np.ones_like(alt),
                    'C_perp': C_perp * np.ones_like(alt),
                    'C_phi': C_phi * np.ones_like(alt),
                    'km1': 1.0 * np.ones_like(alt),
                    'km2': 0.0 * np.ones_like(alt),
                    'km3': 1.0 * np.ones_like(alt),
                    'km4': 1.0 * np.ones_like(alt),
                   }
    
    # Update the simulator set-point
    simulator.update_dict(setpoint_CEM, name='setpoint')

    # Define cost function: penalize the squared error between parallel & perpendicular velocity and heading
    cost = ((simulator.model.x['v_para'] - simulator.model.tvp['v_para_set']) ** 2 +
            (simulator.model.x['v_perp'] - simulator.model.tvp['v_perp_set']) ** 2 +
            (simulator.model.x['phi'] - simulator.model.tvp['phi_set']) ** 2 +
            (simulator.model.x['w'] - simulator.model.tvp['w_set']) ** 2 + 
            (simulator.model.x['zeta'] - simulator.model.tvp['zeta_set']) ** 2
           )

    # Set cost function
    simulator.mpc.set_objective(mterm=cost, lterm=cost)

    # Set input penalty: make this small for accurate state following
    simulator.mpc.set_rterm(u_para=1e-6, u_perp=1e-6, u_phi=1e-6, u_w=0.0, u_zeta=0.0)

    print('one')
    t_sim1, x_sim1, u_sim, y_sim1 = simulator.simulate(x0={'w':[control_w]}, mpc=True, return_full_output=True)  # x0=None
    
    simulator.plot('setpoint')
    simulator.plot('x')
    simulator.plot('y')

    
    t_sim, x_sim, u_sim3, y_sim = simulator.simulate(x0={'w':w_0[0]}, mpc=False, u=u_sim, return_full_output=True)
    
    simulator.plot('setpoint')
    simulator.plot('x')
    simulator.plot('y')

#    trajec = pd.DataFrame()
#    trajec['timestamp'] = t_sim
#    trajec['groundspeed'] = y_sim['g']
#    trajec['groundspeed_angle'] = y_sim['psi']+y_sim['phi']
#    trajec['thrust'] = np.sqrt(u_sim['u_para']**2 + u_sim['u_perp']**2)
#    trajec['thrust_angle'] = np.arctan2(u_sim['u_perp'],u_sim['u_para'])
#    trajec['airspeed'] = y_sim['a']
#    trajec['airspeed_angle'] = y_sim['gamma']+y_sim['phi']
#    trajec['position_x'] = x_sim['x']
#    trajec['position_y'] = x_sim['y']
#    trajec['heading_angle_x'] = np.cos(y_sim['phi'])
#    trajec['heading_angle_y'] = np.sin(y_sim['phi'])


    ## To downsample CEM data
    #trajec = trajec.loc[np.arange(0,len(t_sim1),int(0.01/dt))]
#    trajec.reset_index(drop=True,inplace=True)
    
    ## To upsample CEM data
#    uptraj = pd.DataFrame(np.nan,index=range(0,2*len(t_sim1)),columns = trajec.columns)
#    uptraj.iloc[np.arange(0,2*len(t_sim1),2)] = trajec
#    uptraj.interpolate(method='linear',inplace=True)
    
#    print(uptraj.timestamp)

#    heading_angle_predicted = predict_heading_from_fly_trajectory(uptraj, # uptraj or trajec 
#                                                                  24,
#                                                                  augment_with_time_delay_embedding,
#                                                                  headingPredictor, smooth=True,
#                                                                  **time_augmentation_kwargs)

#    head_pred_unwrap = utils.unwrap_angle(heading_angle_predicted)

    stimulus['time'] = t_sim
    stimulus['obj_id'] = count
    
    stimulus['heading'] = np.round(y_sim['phi'],6)                                  # no heading predictor
    
    #stimulus['heading'] = np.round(head_pred_unwrap[np.arange(0,2*len(t_sim),2)],6)  # downsample from headpredmodel
    
    #stimulus['heading'] = np.nan                                                   # upsample from headpredmodel
    #stimulus['heading'].loc[np.arange(0,len(t_sim1),int(0.01/dt))] = np.round(head_pred_unwrap,6)# upsample from headpredmodel
    #stimulus['heading'].interpolate(method='cubic',inplace=True)                   # upsample from headpredmodel
    
    stimulus['course_dir'] = np.round(y_sim['phi'],6)
    stimulus['groundspeed_angle'] = np.round(y_sim['psi'] + y_sim['phi'],6)
    stimulus['airspeed'] = np.round(y_sim['a'],6)
    stimulus['gamma'] = np.round(y_sim['gamma'] + y_sim['phi'] - stimulus['heading'],6)
    stimulus['gspd'] = np.round(y_sim['r'],6)
    stimulus['psi'] = np.round(y_sim['psi'] + y_sim['phi'] - stimulus['heading'],6)
    stimulus['zeta'] = np.round(x_sim['zeta'],6)
    stimulus['wspd'] = np.round(x_sim['w'],6)
    stimulus['altitude'] = np.round(x_sim['z'],6)
    stimulus['fspd'] = np.round(y_sim['g'],6)
    stimulus['thrust'] = np.sqrt(u_sim['u_para']**2 + u_sim['u_perp']**2)
    stimulus['thrust_angle'] = np.arctan2(u_sim['u_perp'],u_sim['u_para'])
    stimulus['torque'] = u_sim['u_phi']
    stimulus['xpos'] = np.round(x_sim['x'],6)
    stimulus['ypos'] = np.round(x_sim['y'],6)
    stimulus['vpara_input'] = v_para
    stimulus['vperp_input'] = v_perp
    stimulus['phi_input'] = phi
    stimulus['sim1_course_dir'] = np.round(y_sim1['phi'],6)
    stimulus['sim1_xpos'] = np.round(x_sim1['x'],6)
    stimulus['sim1_ypos'] = np.round(x_sim1['y'],6)
    stimulus['sim1_zeta'] = np.round(x_sim1['zeta'],6)
    
    ## print('initphi:',np.round(base,6))

    if count == 0:
        stimulus.to_csv(stimfile)
    else:
        stimulus.to_csv(stimfile,header=False)
        
    ## Model PFN activity during trajectory
    pfnmodel = PFN()

    p_d = pfnmodel.model_param['PFNd']
    p_v = pfnmodel.model_param['PFNv']
    p_pc = pfnmodel.model_param['PFNpc']
    p_a = pfnmodel.model_param['PFNa']
    AF = [-stimulus['gamma'].copy().iloc[0], 100*stimulus['airspeed'].copy().iloc[0]]
    OF = [-stimulus['psi'].copy().iloc[0], 100*stimulus['gspd'].copy().iloc[0]]   # "gspd" is actually of speed
    bump = [-stimulus['heading'].copy().iloc[0], -stimulus['gamma'].copy().iloc[0], 100*stimulus['airspeed'].copy().iloc[0]]
    initcond = BuildSSInitial(p_d, p_v, p_pc, p_a, AF, OF, bump)

    pfnmodel.run(tsim=np.array(stimulus['time']),
                 phi=-np.array(stimulus['heading']),
                 a=100*np.array(stimulus['airspeed']),
                 gamma=-np.array(stimulus['gamma']),
                 g=100*np.array(stimulus['gspd']),
                 psi=-np.array(stimulus['psi']),
                 initcond=initcond)

    inputs = {}

    inputs['obj_id'] = count

    neurons = list(pfnmodel.heatmap.keys())
    labels = {'EPG':['EPG_c1','EPG_c2','EPG_c3','EPG_c4','EPG_c5','EPG_c6','EPG_c7','EPG_c8'],
              'PFNdL_pb':['PFNd_c1','PFNd_c2','PFNd_c3','PFNd_c4','PFNd_c5','PFNd_c6','PFNd_c7','PFNd_c8'],
              'PFNdR_pb':['PFNd_c9','PFNd_c10','PFNd_c11','PFNd_c12','PFNd_c13','PFNd_c14','PFNd_c15','PFNd_c16'],
              'PFNvL_pb':['PFNv_c1','PFNv_c2','PFNv_c3','PFNv_c4','PFNv_c5','PFNv_c6','PFNv_c7','PFNv_c8'],
              'PFNvR_pb':['PFNv_c9','PFNv_c10','PFNv_c11','PFNv_c12','PFNv_c13','PFNv_c14','PFNv_c15','PFNv_c16'],
              'PFNpcL_pb':['PFNpc_c1','PFNpc_c2','PFNpc_c3','PFNpc_c4','PFNpc_c5','PFNpc_c6','PFNpc_c7','PFNpc_c8'],
              'PFNpcR_pb':['PFNpc_c9','PFNpc_c10','PFNpc_c11','PFNpc_c12','PFNpc_c13','PFNpc_c14','PFNpc_c15','PFNpc_c16'],
              'PFNaL_pb':['PFNa_c1','PFNa_c2','PFNa_c3','PFNa_c4','PFNa_c5','PFNa_c6','PFNa_c7','PFNa_c8'],
              'PFNaR_pb':['PFNa_c9','PFNa_c10','PFNa_c11','PFNa_c12','PFNa_c13','PFNa_c14','PFNa_c15','PFNa_c16']   
             }
    for celltype in neurons:
        j = 0
        for label in labels[celltype]:
            inputs[label] = pfnmodel.heatmap[celltype][:,j]
            j+=1

    angle_map = np.arange(-np.pi,np.pi,step=np.pi/4)
    n_angles = angle_map.shape[0]
    n_sim = stimulus['time'].shape[0]

    inputs['wind_c1'] = np.empty((n_sim,))
    inputs['wind_c2'] = np.empty((n_sim,))
    inputs['wind_c3'] = np.empty((n_sim,))
    inputs['wind_c4'] = np.empty((n_sim,))
    inputs['wind_c5'] = np.empty((n_sim,))
    inputs['wind_c6'] = np.empty((n_sim,))
    inputs['wind_c7'] = np.empty((n_sim,))
    inputs['wind_c8'] = np.empty((n_sim,))

    for pt in range(n_sim):
        wind = 0.5 + 0.5 * np.cos(angle_map - stimulus['zeta'].iloc[pt] - np.pi)
        inputs['wind_c1'][pt]=wind[0]
        inputs['wind_c2'][pt]=wind[1]
        inputs['wind_c3'][pt]=wind[2]
        inputs['wind_c4'][pt]=wind[3]
        inputs['wind_c5'][pt]=wind[4]
        inputs['wind_c6'][pt]=wind[5]
        inputs['wind_c7'][pt]=wind[6]
        inputs['wind_c8'][pt]=wind[7]

    print(str(count+1)+' out of '+str(n_traj))

    PFNdf = pd.DataFrame(inputs)
    if count == 0:
        PFNdf.to_csv(PFNfile)
    elif count!=0:
        PFNdf.to_csv(PFNfile,header=False)

    count+=1
        
stimfile.close()
PFNfile.close()